In [ ]:
import json

from langsmith import Client

client = Client()  # Reads LANGSMITH_API_KEY and LANGSMITH_ENDPOINT if set
output_path = "interviewee_ragas_traces.jsonl"

tool_names = [
    "retrieve_project_evidence",
    "get_name",
    "get_self_introduction",
    "get_working_background",
    "get_education",
    "get_skills",
    "get_highlighted_project",
]


def message_content(value):
    if isinstance(value, dict):
        if "content" in value:
            return value["content"]
        if "output" in value:
            return message_content(value["output"])
        if "messages" in value:
            return message_content(value["messages"][-1])
    if isinstance(value, list) and value:
        return message_content(value[-1])
    return value


def as_json(value):
    value = message_content(value)
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return value
    return value


def tool_texts(tool_name, run):
    if run is None:
        return []

    payload = as_json(run.outputs)
    if tool_name == "retrieve_project_evidence":
        return [
            item["text"]
            for item in payload.get("evidence", [])
            if item.get("text")
        ]

    if isinstance(payload, dict) and payload.get("text"):
        return [payload["text"]]
    return []


def first_user_input(run):
    messages = run.inputs.get("messages", [])
    for message in messages:
        if message.get("role") in {"user", "human"}:
            return message_content(message)
    return message_content(run.inputs)


def final_ai_output(run):
    return message_content(run.outputs)


def latency_seconds(root_trace, trace_runs):
    end_times = [run.end_time for run in trace_runs if run.end_time is not None]
    end_time = root_trace.end_time or (max(end_times) if end_times else None)
    if root_trace.start_time is None or end_time is None:
        return None
    return (end_time - root_trace.start_time).total_seconds()


root_traces = list(
    client.list_runs(
        project_name="interview-me",
        is_root=True,
        order="desc",
        limit=33,
    )
)

with open(output_path, "w", encoding="utf-8") as output_file:
    for root_trace in root_traces:
        trace_runs = list(client.list_runs(trace_id=root_trace.id))
        runs_by_name = {run.name: run for run in trace_runs}
        retrieved_context = [
            text
            for tool_name in tool_names
            for text in tool_texts(tool_name, runs_by_name.get(tool_name))
        ]
        record = {
            "user_input": first_user_input(root_trace),
            "response": final_ai_output(root_trace),
            "retrieved_contexts": retrieved_context,
            "latency_seconds": latency_seconds(root_trace, trace_runs),
        }
        output_file.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Wrote {len(root_traces)} traces to {output_path}")

/tmp/ipykernel_18243/2379971416.py:80: DeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead. See https://docs.langchain.com/langsmith/smithdb-sdk-migration-query-runs#runs-query for the migration guide.
  client.list_runs(
/tmp/ipykernel_18243/2379971416.py:90: DeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead. See https://docs.langchain.com/langsmith/smithdb-sdk-migration-query-runs#runs-query for the migration guide.
  trace_runs = list(client.list_runs(trace_id=root_trace.id))


Wrote 33 traces to interviewee_ragas_traces.jsonl
